# 01 — Method validation

**Purpose.** Show that the building blocks are correct *before* any comparison is made:

1. the simulator and the 1D CRR reference behave as theory says;
2. LSM and the neural stopping policy both recover the 1D Bermudan put value;
3. tuned LSM converges to the published 2D/5D max-call references as training paths grow;
4. **implementation check:** the neural method trained in the *paper's own regime* (fresh paths every step, raw inputs + payoff, width d+40, batch norm) reaches the published 5D value.

Settings here are fixed in `CONFIG` in advance and are **not** tuned towards the reference values.
Nothing in this notebook feeds into the main study's choices except as a sanity check.

Run order: **01 → 02 → 03**. Set `MODE` in the first cell. The recovery full scope omits the paper-regime check and limits convergence to 5D at 1k/10k/50k over three seeds; each retained section is timed separately before freezing.

In [1]:
MODE = "quick"   # "quick" = smoke test (minutes, numbers are NOT results) | "full" = research run

import json, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from stopping import *          # all methods live in stopping.py (one file, read top to bottom)

print("Thread limits:", limit_threads(1))   # NumPy/BLAS AND PyTorch single-threaded: fair CPU timing
RUN_PAPER_CHECK = MODE == "quick"  # Full paper-regime implementation check not tested.
from validation_sections import validation_checks, validation_put_fit, validation_convergence, validation_paper, validation_dual
BASE = CONFIG[MODE]["base_seed"]
OUT = results_dir(MODE)                     # results/quick or results/full
print("Mode:", MODE, "| results ->", OUT)
cfg = CONFIG[MODE]["validation"]
CHECK_SEED = 500   # seeds/paths used here are disjoint from the main study (seeds 0-9) and calibration (100+)
cfg

Thread limits: {'torch_threads': 1, 'blas': ['openmp:1', 'openblas:1', 'openblas:1']}
Mode: quick | results -> results/quick


{'put_train': 2000,
 'put_seeds': 1,
 'val': 1000,
 'test': 20000,
 'lsm_budgets': [500, 2000, 10000],
 'nn_settings': {'max_steps': 150, 'lr': 0.001, 'batch': 256},
 'paper_check': {'max_steps': 60, 'batch': 1024}}

## 1. Simulator and reference checks
Exact GBM transitions, risk-neutral drift, correlation, nested training sets within a seed, and CRR tree convergence.

In [ ]:
globals().update(validation_checks())


## 2. 1D Bermudan put: LSM and neural policy against CRR
In 1D sorting is the identity, so the neural input is (log price, payoff). Values are on independent test paths;
the interval is the Monte Carlo interval of the test sample (conditional on the fitted policy).

In [ ]:
globals().update(validation_put_fit(MODE, cfg, BASE, OUT, CHECK_SEED, put, put_ref))


## 3. LSM convergence on the paper's max-calls
Tuned LSM (degree and ridge selected on validation paths) at increasing training budgets for three independent training/validation seeds. Test paths are shared, so seed variation is conditional on that test sample. The plot shows seed means with the average conditional test-sample half-width; the saved rows retain every seed and its own Monte Carlo uncertainty. This is a convergence diagnostic, not proof of monotone improvement or optimality.

Full recovery scope: 5D only, budgets 1k/10k/50k, three seeds and 200k test paths. Quick mode retains its original settings.


In [ ]:
globals().update(validation_convergence(MODE, cfg, BASE, OUT, CHECK_SEED))


## 4. Implementation check in the paper's training regime
Raw log-prices + payoff, two hidden layers of width d+40 with batch norm, Adam, and fresh simulated paths at each step. Settings are fixed in CONFIG before evaluation. We compare the independent test estimate with both BCJ's lower estimate (26.156) and midpoint (26.159), retaining the published interval separately. Agreement supports consistency but cannot certify correctness or attribute every main-study shortfall to data budget. The quick training budget is intentionally too small for a research comparison. Primary-source verification remains pending controller-provided paper evidence.

The full paper-regime implementation check is not tested: it exceeds the per-unit timing budget. `RUN_PAPER_CHECK` is false in full mode.


In [ ]:
if RUN_PAPER_CHECK:
    globals().update(validation_paper(cfg, BASE, OUT, CHECK_SEED))
else:
    print("Paper-regime implementation check not tested (full unit budget).")


## Interpretation limits
Simulator and reference assertions test specific numerical properties. Independent 1D policy estimates are checked against a refined CRR reference, with a heuristic grid band reported separately from sampling error. Three training seeds assess baseline sensitivity conditional on shared test paths. BCJ agreement is a consistency check, not certification. Quick outputs are smoke evidence only; all scientific conclusions require successful full-run evidence. Settings for notebook 03 are selected only in notebook 02.


## 5. Policy-value nested dual: derivation

Nested policy-value dual, following Andersen & Broadie (2004),
Management Science 50(9), 1222–1234, doi:10.1287/mnsc.1040.0258.

For the outer GBM filtration F_k let Z_k=discount(k)*g(S_k),
C_k=E[Z_tau(k+1)|F_k], L_k=Z_k on a policy stop, else C_k,
and L_N=Z_N. Then E[L_(k+1)|F_k]=C_k. Thus M_0=0 and
M_(k+1)-M_k=L_(k+1)-C_k define a martingale. For any stopping
time tau, E[Z_tau-M_tau]=E[Z_tau], hence E[max_{1..N}(Z-M)]
is an upper bound. On continuing dates L_k=C_k, giving telescoping.

Estimate every C_k (including k=0) with fresh independent inner rollouts
of the fixed policy, discounted to zero. Conditional on the WHOLE outer
path, each estimate is unbiased; linearity gives E[Mhat|outer]=M.
Conditional Jensen for the maximum therefore preserves the upper bound
in expectation and adds nonnegative inner bias. Mhat need NOT be a
martingale in a filtration revealing its inner samples at each date:
-C_hat_k is then already known. Mean-zero unconditional increments alone
are not a proof. Neither an individual maximum nor its sample mean is
a guaranteed upper bound. Outer CIs describe sampling error around the
finite-inner expectation; they do not remove inner bias.

Policies must be fixed deterministic state rules, trained independently
of these paths. No inner estimate is used to decide exercise. Primary
paper verification is pending offline source access.


The implementation is in `dual.py`. The independent M3 experiments retain CRR coverage, conditional-drift diagnostics and inner-budget selection evidence. The small experiment below checks notebook integration; quick outputs are smoke evidence only. M3 selected inner=3200 and outer=2000 for the two tested 5D policies; transfer to other policies requires spot checks.

## 6. Executable 1D primal–dual check

Fit on independent training/validation paths, then pair primal and nested dual on fresh outer paths. Report outer sampling error and the CRR grid band separately. The inner sweep diagnoses finite-inner upward bias; two budgets cannot establish its absence. The full-mode 4-SE bracketing diagnostic is an implementation check, not a guaranteed confidence bound or a test of policy optimality. Small quick samples only check execution. The broader M3 coverage and adversarial tests remain necessary evidence.


In [ ]:
globals().update(validation_dual(MODE, cfg, BASE, OUT, CHECK_SEED, put, put_ref))
